# Analisis Temporal de Precios - TFG

Este notebook usa los datos procesados para analizar patrones de dynamic pricing en portatiles de Amazon y PcComponentes.

Bloques principales:
1. Evolucion diaria de precios por plataforma.
2. Ranking de productos con mayor variacion de precio.
3. Resumen ejecutivo para memoria del TFG.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"
candidatos = sorted(PROCESSED_DIR.glob("precios_portatiles_procesado_*.csv"))
if not candidatos:
    raise FileNotFoundError("No hay CSV procesados en data/processed")

latest_csv = candidatos[-1]
print(f"CSV procesado cargado: {latest_csv}")

CSV procesado cargado: /Users/david/Documents/scraper/data/processed/precios_portatiles_procesado_20260330_1303.csv


In [2]:
df = pd.read_csv(latest_csv)
df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], errors="coerce")
df["fecha_dia"] = df["fecha_extraccion"].dt.date

print("Filas:", len(df))
print("Plataformas:", sorted(df["plataforma"].dropna().unique().tolist()))
df.head()

Filas: 49
Plataformas: ['Amazon', 'PcComponentes']


,nombre,precio_actual,precio_original,descuento,valoracion,plataforma,fecha,precio_actual_num,precio_original_num,descuento_pct,fecha_extraccion,anio,mes,dia,fecha_dia
0,"Ordenador portátil de 14,1 Pulgadas, Dual-Core...","199,.99",NaN,NaN,"4,0 de 5 estrellas",Amazon,2026-03-30 13:03,199.99,NaN,NaN,2026-03-30 13:03:00,2026,3,30,2026-03-30
1,PC Portátil 14 Pulgadas Celeron N4000 (hasta 2...,"209,.00",NaN,NaN,"4,0 de 5 estrellas",Amazon,2026-03-30 13:03,209.00,NaN,NaN,2026-03-30 13:03:00,2026,3,30,2026-03-30
2,Ordenador Portátil de 14 Pulgadas Celeron N400...,"209,.00",NaN,NaN,"5,0 de 5 estrellas",Amazon,2026-03-30 13:03,209.00,NaN,NaN,2026-03-30 13:03:00,2026,3,30,2026-03-30
3,"15,6 Pulgadas Ordenador Portatil, Celeron N400...","239,.99",NaN,NaN,"3,4 de 5 estrellas",Amazon,2026-03-30 13:03,239.99,NaN,NaN,2026-03-30 13:03:00,2026,3,30,2026-03-30
4,ASUS Chromebook CX1505CTA - Ordenador Portátil...,"259,.00",NaN,NaN,"4,2 de 5 estrellas",Amazon,2026-03-30 13:03,259.00,NaN,NaN,2026-03-30 13:03:00,2026,3,30,2026-03-30


In [3]:
evolucion = (
    df.groupby(["fecha_dia", "plataforma"], as_index=False)
      .agg(
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          n_productos=("nombre", "count")
      )
)

display(evolucion.sort_values(["fecha_dia", "plataforma"]))

fig = px.line(
    evolucion,
    x="fecha_dia",
    y="precio_medio",
    color="plataforma",
    markers=True,
    title="Evolucion diaria del precio medio por plataforma"
)
fig.update_layout(xaxis_title="Fecha", yaxis_title="Precio medio (EUR)")
fig.show()

,fecha_dia,plataforma,precio_medio,precio_mediano,n_productos
0,2026-03-30,Amazon,431.075833,418.49,24
1,2026-03-30,PcComponentes,957.278800,899.00,25


In [4]:
# Firma de producto para agrupar el mismo item a lo largo del tiempo
df["producto_id"] = (
    df["nombre"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str[:90]
)

variacion = (
    df.groupby(["plataforma", "producto_id"], as_index=False)
      .agg(
          observaciones=("precio_actual_num", "count"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max")
      )
)

variacion["variacion_abs"] = (variacion["precio_max"] - variacion["precio_min"]).round(2)
variacion["variacion_pct"] = ((variacion["variacion_abs"] / variacion["precio_min"]) * 100).round(2)

ranking = variacion.sort_values("variacion_abs", ascending=False).head(20)
display(ranking)

fig_rank = px.bar(
    ranking.head(10),
    x="variacion_abs",
    y="producto_id",
    color="plataforma",
    orientation="h",
    title="Top 10 productos con mayor variacion absoluta"
)
fig_rank.update_layout(xaxis_title="Variacion absoluta (EUR)", yaxis_title="Producto (id abreviado)")
fig_rank.show()

,plataforma,producto_id,observaciones,precio_min,precio_max,variacion_abs,variacion_pct
0,Amazon,"15,6 pulgadas ordenador portatil, celeron n400...",1,239.99,239.99,0.0,0.0
25,PcComponentes,apple macbook air apple m4 10 núcleos/16 gb/25...,1,899.00,899.00,0.0,0.0
27,PcComponentes,apple macbook air apple m4 10 núcleos/16 gb/51...,1,1079.00,1079.00,0.0,0.0
28,PcComponentes,asus v16 v3607vm-rp010 intel core 7 240h/32gb/...,1,1229.00,1229.00,0.0,0.0
29,PcComponentes,asus vivobook go 15 e1504fa-bq2447 amd ryzen 5...,1,444.99,444.99,0.0,0.0
30,PcComponentes,"hp victus 15-fa2005ns 15.6"" intel core 7 240h ...",1,1099.00,1099.00,0.0,0.0
31,PcComponentes,lenovo ideapad slim 3 gen 8 15irh8 intel core ...,1,549.00,549.00,0.0,0.0
32,PcComponentes,msi cyborg 15 b2rwfkg-201xes intel core 7 240h...,1,1299.00,1299.00,0.0,0.0
33,PcComponentes,msi vector 16 hx ai a2xwhg-097xes intel core u...,1,1899.00,1899.00,0.0,0.0
34,PcComponentes,"pccom revolt 5070 intel core i7-14650hx 16""/qh...",1,1658.99,1658.99,0.0,0.0


In [5]:
resumen = (
    df.groupby("plataforma", as_index=False)
      .agg(
          productos=("nombre", "count"),
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max"),
          descuento_medio_pct=("descuento_pct", "mean")
      )
)

resumen = resumen.round(2).sort_values("precio_medio")
display(resumen)

print("Resumen ejecutivo automatico:")
print(f"- Muestra total analizada: {len(df)} observaciones")
for _, row in resumen.iterrows():
    print(
        f"- {row['plataforma']}: n={int(row['productos'])}, precio medio={row['precio_medio']} EUR, "
        f"rango=[{row['precio_min']}, {row['precio_max']}] EUR"
    )

if (variacion['variacion_abs'] > 0).any():
    top = variacion.sort_values('variacion_abs', ascending=False).iloc[0]
    print(
        f"- Mayor variacion detectada: {top['variacion_abs']} EUR en {top['plataforma']} (producto_id abreviado)."
    )
else:
    print("- Aun no hay variacion temporal significativa: se necesita acumular mas dias de extraccion.")

,plataforma,productos,precio_medio,precio_mediano,precio_min,precio_max,descuento_medio_pct
0,Amazon,24,431.08,418.49,199.99,799.0,16.23
1,PcComponentes,25,957.28,899.00,359.00,1899.0,24.80


Resumen ejecutivo automatico:
- Muestra total analizada: 49 observaciones
- Amazon: n=24, precio medio=431.08 EUR, rango=[199.99, 799.0] EUR
- PcComponentes: n=25, precio medio=957.28 EUR, rango=[359.0, 1899.0] EUR
- Aun no hay variacion temporal significativa: se necesita acumular mas dias de extraccion.
